# GB JFSC — Jersey Financial Services Commission

Scrapes the 7 public "Regulated entities / Regulated funds" lists.

The website loads its tables from two Umbraco JSON APIs (no HTML table in the
raw page), so this scraper calls those APIs directly with `requests` — no
Selenium / browser / OCR needed. `PageSize=10000` returns every row in one call.

| ListCode | ListName | API | lawSection refiner |
|---|---|---|---|
| 1 | Regulated Banks | RegulatedEntitiesSearch | bankingdc |
| 2 | Jersey Funds | RegulatedFundsSearch | (none) |
| 3 | Regulated Fund Service Providers | RegulatedEntitiesSearch | fundservicesbusinessfsb |
| 4 | Insurance | RegulatedEntitiesSearch | ibacomposite,ibageneral,ibalongterm,ibbcomposite,ibbgeneral,ibblongterm |
| 5 | Regulated Investment Business | RegulatedEntitiesSearch | investmentbusinessib |
| 6 | Regulated Trust Company Business | RegulatedEntitiesSearch | trustcompanybusinesstcb |
| 7 | Money Service Business | RegulatedEntitiesSearch | moneyservicebusinessmsb |


In [1]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
import os
import datetime
from time import sleep

import requests
import pandas as pd

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'GB JFSC'   ## current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

# scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
scriptfolder = os.path.dirname(os.path.abspath('__file__'))   ## in production use __file__
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')   # if files are downloaded during the process
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

processdate = now.strftime('%Y-%m-%d')


Running GB JFSC Web Scraping Tool v.1.0


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}

# The two JSON endpoints the website calls behind the scenes
BASE_URL = "https://www.jerseyfsc.org"
API_ENT  = "https://www.jerseyfsc.org/umbraco/api/RegulatedEntitiesSearch/GetResults"
API_FUND = "https://www.jerseyfsc.org/umbraco/api/RegulatedFundsSearch/GetResults"

# regdict: regulatorName + ' <ListCode>' -> (api_url, lawSection refiner value or None)
regdict = {
    regulatorName + ' 1': (API_ENT,  'bankingdc'),
    regulatorName + ' 2': (API_FUND, None),
    regulatorName + ' 3': (API_ENT,  'fundservicesbusinessfsb'),
    regulatorName + ' 4': (API_ENT,  'ibacomposite,ibageneral,ibalongterm,ibbcomposite,ibbgeneral,ibblongterm'),
    regulatorName + ' 5': (API_ENT,  'investmentbusinessib'),
    regulatorName + ' 6': (API_ENT,  'trustcompanybusinesstcb'),
    regulatorName + ' 7': (API_ENT,  'moneyservicebusinessmsb'),
}

ListName = {
    regulatorName + ' 1': 'Regulated Banks',
    regulatorName + ' 2': 'Jersey Funds',
    regulatorName + ' 3': 'Regulated Fund Service Providers',
    regulatorName + ' 4': 'Insurance',
    regulatorName + ' 5': 'Regulated Investment Business',
    regulatorName + ' 6': 'Regulated Trust Company Business',
    regulatorName + ' 7': 'Money Service Business',
}

HEADERS = {
    "Content-Type": "application/json;charset=UTF-8",
    "Accept": "application/json, text/plain, */*",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
}

#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key] = sqldict[key] + empty
    return sqldict


def fetch_address(detail_url):
    """Fetch an entity detail page and return its <address> block as one line.

    Each entity page (lists 1, 3-7) has:
        <address><strong>Name</strong><br/>line<br/>line...</address>
    We drop the leading <strong>name</strong> (already captured in 'Name') and
    join the remaining lines with ', '. Returns '' when the page has no address
    block (funds, or the occasional entity with no published address).
    """
    try:
        html = session.get(BASE_URL + detail_url, headers=HEADERS, verify=False, timeout=60).text
    except Exception:
        return ''
    m = re.search(r'<address>(.*?)</address>', html, re.S | re.I)
    if not m:
        return ''
    block = re.sub(r'<strong>.*?</strong>', '', m.group(1), count=1, flags=re.S | re.I)
    block = re.sub(r'(?i)<br\s*/?>', '\n', block)
    block = re.sub(r'<[^>]+>', '', block)
    lines = [re.sub(r'\s+', ' ', ln).strip().rstrip(',').strip() for ln in block.split('\n')]
    return ', '.join(ln for ln in lines if ln)


In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------
session = requests.Session()

for reg in regdict:
    api_url, refiner = regdict[reg]
    body = {"Keyword": "", "PageSize": "10000", "PageNumber": 1, "Refiners": []}
    if refiner:
        body["Refiners"] = [{"Name": "lawSection", "SelectedValue": refiner}]

    r = session.post(api_url, json=body, headers=HEADERS, verify=False, timeout=60)
    r.raise_for_status()
    results = r.json().get("Results", [])
    print(f"[INFO] {reg} ({ListName[reg]}): {len(results)} rows")

    for idx, item in enumerate(results, 1):
        name_ = (item.get("Title") or "").strip()
        if not name_:
            continue
        url_ = item.get("Url") or ""
        m = re.search(r"/(\d+)\s*$", url_)
        jfsc_id = m.group(1) if m else ""
        # Properties is a list of {Name, Value}; licences (entities) or fund type (funds)
        props = {p.get("Name"): p.get("Value") for p in (item.get("Properties") or [])}
        license_ = props.get("LawSection") or props.get("FundType") or ""

        # Address lives only on the entity detail pages (lists 1, 3-7); funds have none.
        address_1 = fetch_address(url_) if api_url == API_ENT else ''

        sqldict['Name'].append(name_)
        sqldict['InternalID_1'].append(jfsc_id)
        sqldict['InternalID_1_type'].append('JFSC Reference')
        sqldict['License_Type'].append(license_)
        sqldict['Address_1'].append(address_1)
        sqldict['RegulationType'].append('Regulated')
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['ListName'].append(ListName[reg])
        sqldict['ListProcessDate'].append(processdate)
        sqldict = bourange_same_length_array(sqldict)

        if api_url == API_ENT and idx % 100 == 0:
            print(f"    ...{idx}/{len(results)} address pages fetched for {reg}")

print(f"[INFO] TOTAL rows collected: {len(sqldict['Name'])}")


In [4]:
#------------------------------------------------ Begin_writer and save df to excel ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df = df[df['Name'] != '']
df.to_excel(filename, 'SQL Ready', index=False)
print(f"Saved {len(df)} rows -> {filename}")


C:\Users\wuj1\AppData\Local\Temp\5\ipykernel_38792\379621782.py:5: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(filename, 'SQL Ready', index=False)


Saved 2098 rows -> GB JFSC SQL Ready 2026-06-15 18.30.17.xlsx
